In [1]:
import openmc
import numpy as np
import pandas as pd

In [2]:
# sp_filename = 'statepointBorated.100.h5'
# sp_filename = 'statepointDHe3.100.h5'
sp_filename = 'statepointDT.100.h5'

In [3]:
sp = openmc.StatePoint(sp_filename)

# access the tally
tally = sp.get_tally(name='dpa_on_tf_coils')

df = tally.get_pandas_dataframe()

print(df)

   cell nuclide          score     mean  std. dev.
0    17   total  damage-energy 2.39e-01   2.40e-02


In [4]:
damage_energy_in_ev = df['mean'].sum() #T_D in NRT equation
E_D = (40 * 0.2862) + (25 * 0.7138) # threshold displacement energy of TF coils in eV
volume_tf_coils = 37789575.821
print('Damage energy deposited per source neutron = ', f"{damage_energy_in_ev:.2f}", 'eV\n')

print('Two times the threshold energy is needed to displace an atom')
displacements_per_source_neutron = 0.8*damage_energy_in_ev / (2*E_D)
print('(NRT) Displacements per source neutron = ', f"{displacements_per_source_neutron:.2f}", '\n')

Damage energy deposited per source neutron =  0.24 eV

Two times the threshold energy is needed to displace an atom
(NRT) Displacements per source neutron =  0.00 



In [5]:
fusion_power = 525e6  # units Watts
energy_per_fusion_reaction = 17.6e6  # units eV
eV_to_Joules = 1.60218e-19  # multiplication factor to convert eV to Joules
number_of_neutrons_per_second = fusion_power / (energy_per_fusion_reaction * eV_to_Joules)
print('Number of neutrons per second', number_of_neutrons_per_second, '\n')

number_of_neutrons_per_year = number_of_neutrons_per_second * 60 * 60 * 24 * 365.25
print('Number of neutrons per full power year ', f"{number_of_neutrons_per_year:.2e}")

displacements_for_all_atoms = number_of_neutrons_per_year * displacements_per_source_neutron
# displacements_for_all_atoms = number_of_neutrons_per_second * displacements_per_source_neutron

print('(NRT) Displacements for all atoms in the volume ', f"{displacements_for_all_atoms:.2e}", '\n')

print('Now the number of atoms in the volume must be found to find displacements per atom (DPA)')


tf_c_mass_in_g = ((88.905*0.1334)+(137.327*0.4123)+(63.546*0.2862)+(16.00*0.1681))*1.66054E-24  # molar mass multiplier by the atomic mass unit (u)
density_of_tf_coils_in_g_per_cm3 = (4.469*0.1334)+(3.51*0.4123)+(8.96*0.2862)+(0.001429*0.1681) # density of TF coils in g/cm^3

number_of_atoms = volume_tf_coils * density_of_tf_coils_in_g_per_cm3 / (tf_c_mass_in_g)

print('Number of TF coils atoms', f"{number_of_atoms:.2e}")

Number of neutrons per second 1.8618098749544654e+20 

Number of neutrons per full power year  5.88e+27
(NRT) Displacements for all atoms in the volume  1.92e+25 

Now the number of atoms in the volume must be found to find displacements per atom (DPA)
Number of TF coils atoms 1.17e+30


In [6]:
DPA_NRT = displacements_for_all_atoms / number_of_atoms

print('NRT-DPA =', f"{DPA_NRT:.2e}")

NRT-DPA = 1.63e-05
